# Task 4 — Valutazione e ottimizzazione dei classificatori manuali

Un notebook per entrambi: il protocollo e' lo stesso e il punto e' il confronto. Motivazioni:
`documentation/04_valutazione.md`.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

SEME = 42

training = pd.read_csv("../data/training.csv").set_index("USERID")
y, X = training["ABBANDONO"], training.drop(columns="ABBANDONO")

manuale = pd.read_csv("../data/manuale.csv").set_index("USERID")
y_man, X_man = manuale["ABBANDONO"], manuale.drop(columns="ABBANDONO")

X_ric, X_ver, y_ric, y_ver = train_test_split(X, y, test_size=.3, stratify=y, random_state=SEME)
CV = StratifiedKFold(5, shuffle=True, random_state=SEME)

print(f"ricerca  : {len(y_ric)} studenti (positivi {y_ric.mean():.1%})")
print(f"verifica : {len(y_ver)} studenti (positivi {y_ver.mean():.1%})")

ricerca  : 4924 studenti (positivi 57.7%)
verifica : 2111 studenti (positivi 57.7%)


## 1. Il protocollo

Provare molte configurazioni e misurarle sugli stessi dati restituisce il **massimo di un insieme di
rumore**, non una stima. Quindi: ricerca sul 70%, verifica sul 30% una volta sola, 5-fold CV per il
confronto finale.

La CV serve soprattutto per la **deviazione standard**: senza, non si puo' dire se mezzo punto sia
un miglioramento.

## 2. Punto di partenza

Riscriviamo le funzioni del Task 2 e controlliamo che riproducano i risultati di allora.

In [2]:
def entropia_binaria(positivi, totali):
    """Entropia in bit dai conteggi: vale per un nodo o per mille nodi in parallelo."""
    positivi, totali = np.asarray(positivi, float), np.asarray(totali, float)
    p = np.divide(positivi, totali, out=np.zeros_like(totali), where=totali > 0)
    q = 1 - p
    with np.errstate(divide="ignore", invalid="ignore"):
        h = -(np.where(p > 0, p * np.log2(p), 0.0) + np.where(q > 0, q * np.log2(q), 0.0))
    return h + 0.0

def entropia(v):
    return float(entropia_binaria((v == 1).sum(), len(v)))

def soglie_candidate(colonna, n_soglie=40):
    """Punti medi fra valori consecutivi (regola del Task 2); se sono troppi, li si campiona."""
    v = np.unique(colonna)
    if len(v) - 1 <= n_soglie:
        return (v[:-1] + v[1:]) / 2
    return np.unique(np.quantile(colonna, np.linspace(.02, .98, n_soglie)))

def miglior_split(Xn, yn, n_soglie=40):
    """Feature e soglia con information gain massimo, valutando tutte le coppie insieme.

    Le ~320 coppie (feature, soglia) del nodo finiscono impilate in un solo vettore, e
    `Xn[:, quale] <= soglie` costruisce in un colpo la matrice campioni x coppie di tutte
    le partizioni candidate. Conteggi e entropie escono da somme di colonna. E' il punto
    piu' caldo del notebook — viene richiamato a ogni nodo di ogni albero di ogni piega —
    e la versione a due cicli annidati impiegava una cinquantina di volte tanto.
    """
    candidate = [soglie_candidate(Xn[c], n_soglie) for c in Xn.columns]
    quale  = np.repeat(np.arange(Xn.shape[1]), [len(s) for s in candidate])
    soglie = np.concatenate(candidate)

    a_sinistra = Xn.to_numpy()[:, quale] <= soglie
    positivi = (yn.to_numpy() == 1)[:, None]
    n = len(yn)
    n_sx, pos_sx = a_sinistra.sum(axis=0), (a_sinistra & positivi).sum(axis=0)

    ig = (entropia(yn) - (n_sx / n) * entropia_binaria(pos_sx, n_sx)
          - ((n - n_sx) / n) * entropia_binaria(int(positivi.sum()) - pos_sx, n - n_sx))
    ig[(n_sx == 0) | (n_sx == n)] = 0.0        # split che non separano nulla: scartati

    i = int(np.argmax(ig))                     # a pari merito argmax tiene la prima coppia
    if ig[i] <= 0:
        return 0.0, None, None
    return float(ig[i]), Xn.columns[quale[i]], float(soglie[i])

def cresci(Xn, yn, prof=0, max_prof=3, min_campioni=20):
    if entropia(yn) == 0 or prof == max_prof or len(yn) < min_campioni:
        return {"foglia": int(yn.mode().iloc[0]), "n": len(yn)}
    g, c, th = miglior_split(Xn, yn)
    if c is None or g <= 1e-9:
        return {"foglia": int(yn.mode().iloc[0]), "n": len(yn)}
    m = Xn[c] <= th
    return {"feature": c, "soglia": th, "ig": g, "n": len(yn),
            "si": cresci(Xn[m], yn[m], prof + 1, max_prof, min_campioni),
            "no": cresci(Xn[~m], yn[~m], prof + 1, max_prof, min_campioni)}

def predici_albero(radice, Xn):
    """Scende l'albero a maschere: la ricorsione e' sui nodi, non sulle righe."""
    previsioni = np.empty(len(Xn), dtype=int)

    def scendi(nodo, quali):
        if "foglia" in nodo:
            previsioni[quali] = nodo["foglia"]
            return
        a_sinistra = quali & (Xn[nodo["feature"]].to_numpy() <= nodo["soglia"])
        scendi(nodo["si"], a_sinistra)
        scendi(nodo["no"], quali & ~a_sinistra)

    scendi(radice, np.ones(len(Xn), dtype=bool))
    return pd.Series(previsioni, index=Xn.index)

def nb_addestra(Xb, yb, n_valori=2, alpha=1.0):
    """Naive Bayes su feature discretizzate, con correzione di Laplace.

    I conteggi di tutte le terne (classe, feature, valore) escono da un solo `bincount`:
    si appiattisce la terna in un indice unico e si contano le occorrenze in una passata,
    invece di estrarre un sottoinsieme per ciascuna delle 16+ combinazioni.
    """
    A, k_riga, n_f = Xb.to_numpy(), yb.to_numpy(), Xb.shape[1]
    piatto = (k_riga[:, None] * n_f + np.arange(n_f)) * n_valori + A
    conteggi = np.bincount(piatto.ravel(), minlength=2 * n_f * n_valori).reshape(2, n_f, n_valori)
    per_classe = np.bincount(k_riga, minlength=2)
    return per_classe / len(yb), (conteggi + alpha) / (per_classe[:, None, None] + n_valori * alpha)

def nb_predici(Xb, modello):
    """Somma i log delle condizionate per tutte le righe insieme, con indicizzazione avanzata."""
    priori, cond = modello
    A = Xb.to_numpy()
    lp = np.log(priori)[:, None] + np.log(cond)[:, np.arange(A.shape[1]), A].sum(axis=2)
    return (lp[1] > lp[0]).astype(int)

# controprova: le funzioni riproducono i risultati del Task 2 sui 12 campioni?
albero_t2 = cresci(X_man, y_man, max_prof=3, min_campioni=2)
SOGLIE_MAN = X_man.median()
nb_t2 = nb_addestra((X_man > SOGLIE_MAN).astype(int), y_man)
print("albero: feature e soglia del Task 2 :", albero_t2["feature"], "<=", round(albero_t2["soglia"], 1))
print("albero: accuratezza su manuale.csv  :", accuracy_score(y_man, predici_albero(albero_t2, X_man)))
print("NB    : accuratezza su manuale.csv  :", accuracy_score(y_man, nb_predici((X_man > SOGLIE_MAN).astype(int), nb_t2)))

albero: feature e soglia del Task 2 : n_attivita_distinte <= 22.5
albero: accuratezza su manuale.csv  : 1.0
NB    : accuratezza su manuale.csv  : 1.0


In [3]:
def cv_punteggi(fai_predizione):
    """Accuratezza e F1 su 5 pieghe stratificate di training.csv."""
    acc, f1 = [], []
    for i_ric, i_ver in CV.split(X, y):
        Xa, ya, Xb, yb = X.iloc[i_ric], y.iloc[i_ric], X.iloc[i_ver], y.iloc[i_ver]
        p = fai_predizione(Xa, ya, Xb)
        acc.append(accuracy_score(yb, p)); f1.append(f1_score(yb, p))
    return np.array(acc), np.array(f1)

risultati = {}
def registra(nome, funzione):
    a, f = cv_punteggi(funzione)
    risultati[nome] = a
    print(f"{nome:<44} acc {a.mean():.4f} +/- {a.std():.4f}   F1 {f.mean():.4f}")

print("=== i due classificatori del Task 2, applicati tali e quali ===")
registra("albero Task 2 (soglia 22,5 dai 12 campioni)",
         lambda Xa, ya, Xb: (Xb.n_attivita_distinte <= 22.5).astype(int))
registra("Naive Bayes Task 2 (mediane di manuale.csv)",
         lambda Xa, ya, Xb: nb_predici((Xb > SOGLIE_MAN).astype(int),
                                       nb_addestra((Xa > SOGLIE_MAN).astype(int), ya)))

=== i due classificatori del Task 2, applicati tali e quali ===
albero Task 2 (soglia 22,5 dai 12 campioni)  acc 0.7734 +/- 0.0088   F1 0.8031
Naive Bayes Task 2 (mediane di manuale.csv)  acc 0.7902 +/- 0.0075   F1 0.8184


## 3. Ottimizzazione dell'albero

### 3.1 La soglia

Nel Task 2 valeva 22,5, stimata su 12 campioni. Cerchiamola sui 4.924 dell'insieme di ricerca.

In [4]:
# le 59 soglie provate insieme: matrice studenti x soglie, poi una media di colonna
griglia = np.arange(2, 61)
previsto = X_ric.n_attivita_distinte.to_numpy()[:, None] <= griglia
punteggi = (previsto == (y_ric.to_numpy()[:, None] == 1)).mean(axis=0)
soglia_ott = int(griglia[int(np.argmax(punteggi))])

print(f"soglia migliore sull'insieme di ricerca: {soglia_ott} (accuratezza {punteggi.max():.4f})")
for nome, s in [("Task 2", 22.5), ("ottimizzata", soglia_ott)]:
    p = (X_ver.n_attivita_distinte <= s).astype(int)
    print(f"  soglia {str(s):>4} ({nome:<11}) sulla verifica: acc {accuracy_score(y_ver, p):.4f}  F1 {f1_score(y_ver, p):.4f}")

soglia migliore sull'insieme di ricerca: 28 (accuratezza 0.7837)
  soglia 22.5 (Task 2     ) sulla verifica: acc 0.7603  F1 0.7936
  soglia   28 (ottimizzata) sulla verifica: acc 0.7740  F1 0.8167


### 3.2 La feature

Ogni feature come nodo singolo, in **entrambe le direzioni**: per alcune l'abbandono sta sopra la
soglia, non sotto.

In [5]:
def stump(colonna, soglia, verso):
    return (colonna <= soglia).astype(int) if verso == "<=" else (colonna > soglia).astype(int)

def accuratezze(colonna, y, soglie):
    """Accuratezza di `colonna <= soglia` e di `colonna > soglia`, per tutte le soglie insieme."""
    previsto = colonna.to_numpy()[:, None] <= soglie          # (campioni, soglie)
    vero = (y.to_numpy() == 1)[:, None]
    return (previsto == vero).mean(axis=0), (~previsto == vero).mean(axis=0)

righe = []
for c in X.columns:          # un giro per feature: dentro, le 120 regole sono valutate insieme
    soglie = np.unique(np.quantile(X_ric[c], np.linspace(.02, .98, 60)))
    # entrambe le direzioni: per alcune feature l'abbandono sta sopra la soglia, non sotto
    acc_le, acc_gt = accuratezze(X_ric[c], y_ric, soglie)
    prove = pd.DataFrame({"acc": np.concatenate([acc_le, acc_gt]),
                          "soglia": np.tile(soglie, 2),
                          "verso": np.repeat(["<=", ">"], len(soglie))})
    m = prove.sort_values(["acc", "soglia", "verso"], kind="stable").iloc[-1]
    righe.append({"feature": c, "regola": f"{m.verso} {m.soglia:.3f}", "acc ricerca": round(m.acc, 4),
                  "acc verifica": round(accuracy_score(y_ver, stump(X_ver[c], m.soglia, m.verso)), 4)})
pd.DataFrame(righe).sort_values("acc ricerca", ascending=False).reset_index(drop=True)

,feature,regola,acc ricerca,acc verifica
0,n_azioni,<= 55.000,0.7912,0.7693
1,n_attivita_distinte,<= 28.000,0.7837,0.7740
2,n_giorni_attivi,<= 4.000,0.7807,0.7518
3,durata_giorni,<= 14.193,0.7581,0.7513
4,feature0_media,<= -0.160,0.7053,0.6921
5,feature3_media,<= -0.067,0.6363,0.6324
6,feature2_media,> -0.023,0.6054,0.5973
7,feature1_media,<= 0.518,0.5674,0.5694


**Fregatura.** La feature migliore *in ricerca* e' `n_azioni`, ma sulla verifica le prime due si
invertono. Chi vince la ricerca perde il controllo — ed e' il motivo per cui le due misure vanno
tenute separate.

### 3.3 La profondita'

In [6]:
print(f"{'max_depth':>10} {'acc media':>11} {'dev.std':>9} {'F1 medio':>10}")
albero_cv = {}
for d in (1, 2, 3, 4, 5):
    a, f = cv_punteggi(lambda Xa, ya, Xb, d=d: predici_albero(cresci(Xa, ya, max_prof=d), Xb))
    albero_cv[d] = a
    print(f"{d:>10} {a.mean():11.4f} {a.std():9.4f} {f.mean():10.4f}")
risultati["albero ricresciuto sui dati (profondita' 5)"] = albero_cv[5]
risultati["albero ricresciuto sui dati (profondita' 4)"] = albero_cv[4]

 max_depth   acc media   dev.std   F1 medio


         1      0.7831    0.0086     0.8162


         2      0.7831    0.0086     0.8162


         3      0.7842    0.0097     0.8177


         4      0.7893    0.0095     0.8235


         5      0.7906    0.0141     0.8228


In [7]:
def foglie(n):
    return [n["foglia"]] if "foglia" in n else foglie(n["si"]) + foglie(n["no"])

def conta_nodi(n):
    if "foglia" in n:
        return 0, 0
    sinistra, destra = set(foglie(n["si"])), set(foglie(n["no"]))
    cosmetico = int(len(sinistra) == 1 and len(destra) == 1 and sinistra == destra)
    a1, b1 = conta_nodi(n["si"]); a2, b2 = conta_nodi(n["no"])
    return 1 + a1 + a2, cosmetico + b1 + b2

albero5 = cresci(X_ric, y_ric, max_prof=5)
nodi, cosmetici = conta_nodi(albero5)
print(f"albero di profondita' 5: {nodi} nodi di decisione, {len(foglie(albero5))} foglie")
print(f"nodi in cui entrambi i rami portano alla STESSA classe: {cosmetici} ({cosmetici/nodi:.0%})")
print(f"\nprimo livello: {albero5['feature']} <= {albero5['soglia']:.1f}  (IG {albero5['ig']:.4f})")
print(f"secondo livello: {albero5['si']['feature']} <= {albero5['si']['soglia']:.3f}  (IG {albero5['si']['ig']:.4f})")

albero di profondita' 5: 29 nodi di decisione, 30 foglie
nodi in cui entrambi i rami portano alla STESSA classe: 16 (55%)

primo livello: n_azioni <= 50.0  (IG 0.2451)
secondo livello: durata_giorni <= 0.948  (IG 0.0456)


**Il guadagno c'e' ma meta' dell'albero non serve.** Da profondita' 1 a 5 meno di 0,8 punti, e la
deviazione standard cresce di oltre il 60% (0,0086 → 0,0141): sovradattamento. Il conteggio dei nodi
lo spiega — in **16 nodi su 29 (55%)** entrambi i rami portano alla stessa classe.

## 4. Ottimizzazione del Naive Bayes

### 4.1 Numero di intervalli

In [8]:
def nb_quantili(Xa, ya, Xb, n_bin, alpha=1.0):
    bordi = {c: np.unique(np.quantile(Xa[c], np.linspace(0, 1, n_bin + 1))[1:-1]) for c in Xa.columns}
    A = pd.DataFrame({c: np.digitize(Xa[c], bordi[c]) for c in Xa.columns})
    B = pd.DataFrame({c: np.digitize(Xb[c], bordi[c]) for c in Xb.columns})
    v = int(max(A.max().max(), B.max().max())) + 1
    return nb_predici(B, nb_addestra(A, ya.reset_index(drop=True), n_valori=v, alpha=alpha))

print(f"{'intervalli':>11} {'acc media':>11} {'dev.std':>9}")
for n_bin in (2, 3, 4, 5, 8):
    a, _ = cv_punteggi(lambda Xa, ya, Xb, n=n_bin: nb_quantili(Xa, ya, Xb, n))
    print(f"{n_bin:>11} {a.mean():11.4f} {a.std():9.4f}")
    if n_bin == 4:
        risultati["Naive Bayes, 4 intervalli sui quantili"] = a

 intervalli   acc media   dev.std
          2      0.7750    0.0082


          3      0.7811    0.0113
          4      0.7822    0.0088


          5      0.7819    0.0097
          8      0.7790    0.0113


### 4.2 Da dove prendere le soglie

Il Naive Bayes tarato su **dodici** campioni batte quelli con soglie stimate su migliaia di righe.
Ipotesi: `manuale.csv` e' bilanciato 6/6, `training.csv` ha il 57,7% di positivi, e la mediana di una
popolazione sbilanciata taglia in un punto peggiore. Se e' vero, **ribilanciare deve recuperare il
divario**.

In [9]:
def soglie_da(Xa, ya, modo):
    if modo == "manuale":     return SOGLIE_MAN
    if modo == "training":    return Xa.median()
    if modo == "bilanciato":
        minor = Xa[(ya == 0).to_numpy()]
        magg = Xa[(ya == 1).to_numpy()].sample(len(minor), random_state=SEME)
        return pd.concat([minor, magg]).median()
    if modo == "medie":       return (Xa[(ya == 0).to_numpy()].mean() + Xa[(ya == 1).to_numpy()].mean()) / 2

def nb_con_soglie(Xa, ya, Xb, modo):
    S = soglie_da(Xa, ya, modo)
    return nb_predici((Xb > S).astype(int), nb_addestra((Xa > S).astype(int), ya))

etichette = {"manuale": "mediane di manuale.csv (Task 2)", "training": "mediane di training.csv",
             "bilanciato": "mediane di training ribilanciato", "medie": "punto medio fra le medie di classe"}
per_piega = {}
print(f"{'origine delle soglie':<36} {'acc media':>11} {'dev.std':>9}")
for modo, nome in etichette.items():
    a, _ = cv_punteggi(lambda Xa, ya, Xb, m=modo: nb_con_soglie(Xa, ya, Xb, m))
    per_piega[nome] = a
    print(f"{nome:<36} {a.mean():11.4f} {a.std():9.4f}")
risultati["Naive Bayes, soglie dalle medie di classe"] = per_piega[etichette["medie"]]

base = per_piega[etichette["manuale"]]
print("\nconfronto appaiato, piega per piega, contro le mediane di manuale.csv:")
for nome, a in per_piega.items():
    if nome == etichette["manuale"]:
        continue
    d = base - a
    print(f"  vs {nome:<36} {d.mean():+.4f}   manuale.csv vince {int((d > 0).sum())}/5 pieghe")

origine delle soglie                   acc media   dev.std


mediane di manuale.csv (Task 2)           0.7902    0.0075
mediane di training.csv                   0.7775    0.0086


mediane di training ribilanciato          0.7889    0.0076
punto medio fra le medie di classe        0.7933    0.0080

confronto appaiato, piega per piega, contro le mediane di manuale.csv:
  vs mediane di training.csv              +0.0127   manuale.csv vince 5/5 pieghe
  vs mediane di training ribilanciato     +0.0013   manuale.csv vince 2/5 pieghe
  vs punto medio fra le medie di classe   -0.0031   manuale.csv vince 2/5 pieghe


Ipotesi confermata: le mediane di `training.csv` perdono 5 pieghe su 5, ma appena si ribilancia il
divario sparisce. Per una soglia di taglio conta il **bilanciamento** del campione, non la
numerosita'.

## 5. Confronto finale

In [10]:
finale = pd.DataFrame({
    "accuratezza media": {k: v.mean() for k, v in risultati.items()},
    "dev. standard":     {k: v.std()  for k, v in risultati.items()},
}).sort_values("accuratezza media", ascending=False)
finale["scarto dal migliore"] = finale["accuratezza media"].max() - finale["accuratezza media"]
print(finale.round(4).to_string())

migliore, peggiore = finale.index[0], "albero Task 2 (soglia 22,5 dai 12 campioni)"
d = risultati[migliore] - risultati[peggiore]
print(f"\nmiglioramento complessivo: {d.mean():+.4f} ({d.mean()*100:+.1f} punti), "
      f"ottenuto in {int((d > 0).sum())}/5 pieghe")
print(f"deviazione standard tipica fra le pieghe: {np.mean([v.std() for v in risultati.values()]):.4f}")
print(f"baseline 'sempre abbandono': {y.mean():.4f}")

                                             accuratezza media  dev. standard  scarto dal migliore
Naive Bayes, soglie dalle medie di classe               0.7933         0.0080               0.0000
albero ricresciuto sui dati (profondita' 5)             0.7906         0.0141               0.0027
Naive Bayes Task 2 (mediane di manuale.csv)             0.7902         0.0075               0.0031
albero ricresciuto sui dati (profondita' 4)             0.7893         0.0095               0.0040
Naive Bayes, 4 intervalli sui quantili                  0.7822         0.0088               0.0111
albero Task 2 (soglia 22,5 dai 12 campioni)             0.7734         0.0088               0.0199

miglioramento complessivo: +0.0199 (+2.0 punti), ottenuto in 5/5 pieghe
deviazione standard tipica fra le pieghe: 0.0095
baseline 'sempre abbandono': 0.5771


## 6. Limiti

- **+2,0 punti**, vinti in tutte le pieghe, ma con deviazione standard tipica 0,0095 valgono poco
  piu' di due deviazioni standard: reale, modesto.
- **Il tetto e' ~79%.** Tutte le configurazioni provate cadono in due punti. Non e' un limite
  dell'ottimizzazione ma delle feature: tre su otto sono rumore, le altre cinque misurano la stessa
  cosa.
- **Per il Task 5** serve piu' capacita' di modello, non piu' regolazione fine. Il protocollo resta
  questo.